# 05 - Generación de tiles Sentinel-2 para estaciones con filtro SCL

Este notebook genera tiles `64 x 64 x 12` para las celdas de grilla asociadas a estaciones DAGMA y las fechas Sentinel-2 disponibles.

La política de calidad replica la base usada en Situación 2 y agrega un umbral explícito de nube/sombra:

```text
window_inside == True
shape_ok == True
finite == True
zero_pct < 99
image_std > 1e-6
scl_cloud_shadow_pct <= 30
```

Se guardan tiles aceptados y metadata de todos los intentos, incluyendo rechazados.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform as rio_transform
from rasterio.windows import Window, from_bounds

os.environ.setdefault('GDAL_CACHEMAX', '64')
os.environ.setdefault('OPJ_NUM_THREADS', '1')
os.environ.setdefault('GDAL_NUM_THREADS', '1')

BASE = Path('/workspace/geovision-cali-hf')
RAW = BASE / 'data/sentinel2-sentinel-5p/sentinel2p-raw'
OUT = BASE / 'outputs/situacion3/rubrica/05_tiles_estaciones_filtrados_scl'
IMG_DIR = OUT / 'images'
SCL_DIR = OUT / 'scl'
OUT.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)
SCL_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'station_to_grid': BASE / 'outputs/situacion3/02_grilla_s2_features/station_to_s2_grid_mapping.csv',
    'grid': BASE / 'outputs/situacion3/02_grilla_s2_features/grid_cali_005deg.parquet',
    's2_features': BASE / 'outputs/situacion3/02_grilla_s2_features/s2_grid_features_with_quality.parquet',
    'scene_inventory': BASE / 'outputs/situacion3/rubrica/05_auditoria_fuentes_raster_sentinel2/raw_s2_scene_completeness.csv',
}

TILE_SIZE = 64
BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
SCL_CLOUD_SHADOW = {3, 8, 9, 10}
SCL_VALID_VISUAL = {4, 5, 6, 7}
SCL_INVALID_EXTRA = {0, 1, 2, 11}
MAX_SCL_CLOUD_SHADOW_PCT = 30.0

def utc_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if np.isnan(float(value)) else float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    return str(value)


## Inventario de escenas y funciones de lectura


In [2]:
def find_band_file(scene_dir, band):
    matches = sorted(Path(scene_dir).rglob(f'*_{band}_*.jp2'))
    if not matches:
        raise FileNotFoundError(f'No hay banda {band} en {scene_dir}')
    return matches[0]

def find_scl_file(scene_dir):
    matches = sorted(Path(scene_dir).rglob('*_SCL_*.jp2'))
    if not matches:
        raise FileNotFoundError(f'No hay SCL en {scene_dir}')
    return matches[0]

def inspect_point(scene_row, lon, lat):
    band_path = find_band_file(scene_row.scene_dir, 'B04')
    try:
        with rasterio.Env(GDAL_CACHEMAX=64, OPJ_NUM_THREADS='1', GDAL_NUM_THREADS='1'):
            with rasterio.open(band_path) as src:
                xs, ys = rio_transform('EPSG:4326', src.crs, [lon], [lat])
                row_px, col_px = src.index(xs[0], ys[0])
                row_off = int(row_px - TILE_SIZE // 2)
                col_off = int(col_px - TILE_SIZE // 2)
                inside = row_off >= 0 and col_off >= 0 and row_off + TILE_SIZE <= src.height and col_off + TILE_SIZE <= src.width
                return {
                    'window_inside': bool(inside),
                    'reason': 'ok' if inside else 'outside_or_edge',
                    'raster_crs': str(src.crs),
                    'raster_width': int(src.width),
                    'raster_height': int(src.height),
                    'row_px': int(row_px),
                    'col_px': int(col_px),
                    'row_off': row_off,
                    'col_off': col_off,
                }
    except Exception as exc:
        return {'window_inside': False, 'reason': f'{type(exc).__name__}: {exc}'}

def tile_stats(image, scl):
    red = image[:, :, 3]
    green = image[:, :, 2]
    nir = image[:, :, 7]
    swir = image[:, :, 10]
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndbi = (swir - nir) / (swir + nir + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)
    return {
        'image_min': float(np.nanmin(image)),
        'image_max': float(np.nanmax(image)),
        'image_mean': float(np.nanmean(image)),
        'image_std': float(np.nanstd(image)),
        'ndvi_mean': float(np.nanmean(ndvi)),
        'ndbi_mean': float(np.nanmean(ndbi)),
        'ndwi_mean': float(np.nanmean(ndwi)),
        'scl_cloud_shadow_pct': float(np.isin(scl, list(SCL_CLOUD_SHADOW)).mean() * 100),
        'scl_valid_visual_pct': float(np.isin(scl, list(SCL_VALID_VISUAL)).mean() * 100),
        'scl_invalid_extra_pct': float(np.isin(scl, list(SCL_INVALID_EXTRA)).mean() * 100),
        'zero_pct': float((image == 0).mean() * 100),
        'high_reflectance_pct': float((image > 1.2).mean() * 100),
    }

def read_tile(scene_row, row_off, col_off):
    scene_dir = Path(scene_row.scene_dir)
    b04_path = find_band_file(scene_dir, 'B04')
    ref_window = Window(col_off=int(col_off), row_off=int(row_off), width=TILE_SIZE, height=TILE_SIZE)
    with rasterio.open(b04_path) as ref_src:
        ref_bounds = rasterio.windows.bounds(ref_window, ref_src.transform)
        ref_crs = ref_src.crs
    arrays = []
    for band in BANDS:
        band_path = find_band_file(scene_dir, band)
        with rasterio.open(band_path) as src:
            if src.crs != ref_crs:
                raise ValueError(f'CRS mismatch {band}: {src.crs} != {ref_crs}')
            win = from_bounds(*ref_bounds, transform=src.transform)
            arr = src.read(1, window=win, out_shape=(TILE_SIZE, TILE_SIZE), resampling=Resampling.bilinear).astype('float32') / 10000.0
            arrays.append(arr)
    image = np.stack(arrays, axis=-1).astype('float32')
    scl_path = find_scl_file(scene_dir)
    with rasterio.open(scl_path) as src:
        win = from_bounds(*ref_bounds, transform=src.transform)
        scl = src.read(1, window=win, out_shape=(TILE_SIZE, TILE_SIZE), resampling=Resampling.nearest).astype('uint8')
    return image, scl, tile_stats(image, scl)


## Carga de estaciones, fechas y escenas


In [3]:
stations = pd.read_csv(PATHS['station_to_grid'])
grid = pd.read_parquet(PATHS['grid'])
s2_dates = pd.read_parquet(PATHS['s2_features'], columns=['date'])
s2_dates['date'] = pd.to_datetime(s2_dates['date']).dt.normalize()
dates = sorted(s2_dates['date'].drop_duplicates())
scene_inventory = pd.read_csv(PATHS['scene_inventory'])
scene_inventory = scene_inventory[scene_inventory['complete_12band_scl'].astype(bool)].copy()
scene_inventory['date'] = pd.to_datetime(scene_inventory['date']).dt.normalize()

station_cells = stations[['estacion', 'nearest_grid_id', 'grid_lat', 'grid_lon', 'distance_km']].drop_duplicates().copy()
station_cells = station_cells.rename(columns={'nearest_grid_id': 'grid_id', 'grid_lat': 'lat', 'grid_lon': 'lon', 'distance_km': 'station_to_grid_distance_km'})

summary_inputs = {
    'n_station_cells': int(len(station_cells)),
    'n_dates': int(len(dates)),
    'n_expected_station_date_tiles': int(len(station_cells) * len(dates)),
    'n_complete_scenes': int(len(scene_inventory)),
    'n_scene_dates': int(scene_inventory['date'].nunique()),
    'date_min': str(min(dates).date()),
    'date_max': str(max(dates).date()),
}
summary_inputs


{'n_station_cells': 9,
 'n_dates': 129,
 'n_expected_station_date_tiles': 1161,
 'n_complete_scenes': 198,
 'n_scene_dates': 131,
 'date_min': '2020-01-02',
 'date_max': '2024-12-16'}

## Generación de tiles

Para cada celda de estación y fecha se busca una escena Sentinel-2 que contenga el punto. Si hay varias, se usa la primera ventana válida.


In [4]:
records = []
total = len(station_cells) * len(dates)
counter = 0
LOG_PATH = OUT / 'generation_progress.log'
LOG_PATH.write_text('', encoding='utf-8')

def log_progress(message, payload=None):
    payload = payload or {}
    line = json.dumps({'time': utc_now(), 'message': message, **payload}, ensure_ascii=False)
    with LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(line + '\\n')
    print(line, flush=True)

def records_summary():
    if not records:
        return {'records': 0, 'accepted': 0, 'rejected': 0}
    tmp = pd.DataFrame(records)
    accepted_count = int(tmp.get('accepted_s2_policy', pd.Series(dtype=bool)).fillna(False).astype(bool).sum())
    reason_counts = tmp.get('reject_reason', pd.Series(dtype=str)).fillna('missing').value_counts().head(8).to_dict()
    return {
        'records': int(len(tmp)),
        'accepted': accepted_count,
        'rejected': int(len(tmp) - accepted_count),
        'top_reject_reasons': {str(k): int(v) for k, v in reason_counts.items()},
    }

log_progress('inicio_generacion_tiles', {'total_station_date_pairs': total, 'n_dates': len(dates), 'n_station_cells': len(station_cells)})

def make_base_rec(cell, date):
    return {
        'tile_id': f"s3station_{cell.grid_id}_{date.strftime('%Y%m%d')}",
        'estacion': cell.estacion,
        'grid_id': cell.grid_id,
        'lat': float(cell.lat),
        'lon': float(cell.lon),
        'date_day': date.date().isoformat(),
        'station_to_grid_distance_km': float(cell.station_to_grid_distance_km),
        'created_at_utc': utc_now(),
    }

def point_window_from_ref(ref_src, lon, lat):
    xs, ys = rio_transform('EPSG:4326', ref_src.crs, [lon], [lat])
    row_px, col_px = ref_src.index(xs[0], ys[0])
    row_off = int(row_px - TILE_SIZE // 2)
    col_off = int(col_px - TILE_SIZE // 2)
    inside = row_off >= 0 and col_off >= 0 and row_off + TILE_SIZE <= ref_src.height and col_off + TILE_SIZE <= ref_src.width
    return inside, row_px, col_px, row_off, col_off

def read_scl_window(scl_src, ref_bounds):
    win = from_bounds(*ref_bounds, transform=scl_src.transform)
    return scl_src.read(1, window=win, out_shape=(TILE_SIZE, TILE_SIZE), resampling=Resampling.nearest).astype('uint8')

def read_image_window(open_bands, ref_bounds):
    arrays = []
    for band in BANDS:
        src = open_bands[band]
        win = from_bounds(*ref_bounds, transform=src.transform)
        arr = src.read(1, window=win, out_shape=(TILE_SIZE, TILE_SIZE), resampling=Resampling.bilinear).astype('float32') / 10000.0
        arrays.append(arr)
    return np.stack(arrays, axis=-1).astype('float32')

with rasterio.Env(GDAL_CACHEMAX=256, OPJ_NUM_THREADS='1', GDAL_NUM_THREADS='1'):
    for date_idx, date in enumerate(dates, start=1):
        date = pd.Timestamp(date).normalize()
        log_progress('inicio_fecha', {'date_idx': date_idx, 'n_dates': len(dates), 'date': date.date().isoformat(), **records_summary()})
        scenes = scene_inventory[scene_inventory['date'].eq(date)].sort_values(['mgrs_tile', 'scene_id']).copy()
        cells_for_date = list(station_cells.itertuples(index=False))

        if scenes.empty:
            for cell in cells_for_date:
                records.append(make_base_rec(cell, date) | {'accepted_s2_policy': False, 'reject_reason': 'no_scene_for_date'})
                counter += 1
            log_progress('sin_escena_fecha', {'date': date.date().isoformat(), **records_summary()})
            continue

        for scene_row in scenes.itertuples(index=False):
            scene_dir = Path(scene_row.scene_dir)
            b04_path = find_band_file(scene_dir, 'B04')
            scl_path_scene = find_scl_file(scene_dir)

            with rasterio.open(b04_path) as ref_src, rasterio.open(scl_path_scene) as scl_src:
                remaining_cells = []
                for cell in cells_for_date:
                    counter += 1
                    if counter == 1 or counter % 100 == 0:
                        print(json.dumps({'stage': 'generate_station_tiles_fast', 'i': counter, 'n': total}, ensure_ascii=False), flush=True)

                    base_rec = make_base_rec(cell, date)
                    try:
                        inside, row_px, col_px, row_off, col_off = point_window_from_ref(ref_src, float(cell.lon), float(cell.lat))
                    except Exception as exc:
                        remaining_cells.append(cell)
                        continue
                    if not inside:
                        remaining_cells.append(cell)
                        continue

                    ref_window = Window(col_off=int(col_off), row_off=int(row_off), width=TILE_SIZE, height=TILE_SIZE)
                    ref_bounds = rasterio.windows.bounds(ref_window, ref_src.transform)
                    rec = base_rec | {
                        'scene_id': scene_row.scene_id,
                        'scene_dir': scene_row.scene_dir,
                        'mgrs_tile': scene_row.mgrs_tile,
                        'row_px': int(row_px),
                        'col_px': int(col_px),
                        'row_off': int(row_off),
                        'col_off': int(col_off),
                        'raster_crs': str(ref_src.crs),
                    }

                    img_path = IMG_DIR / f"{rec['tile_id']}.npy"
                    out_scl_path = SCL_DIR / f"{rec['tile_id']}_scl.npy"
                    try:
                        if img_path.exists() and out_scl_path.exists():
                            image = np.load(img_path).astype('float32')
                            scl = np.load(out_scl_path).astype('uint8')
                            stats = tile_stats(image, scl)
                        else:
                            scl = read_scl_window(scl_src, ref_bounds)
                            scl_stats_only = {
                                'scl_cloud_shadow_pct': float(np.isin(scl, list(SCL_CLOUD_SHADOW)).mean() * 100),
                                'scl_valid_visual_pct': float(np.isin(scl, list(SCL_VALID_VISUAL)).mean() * 100),
                                'scl_invalid_extra_pct': float(np.isin(scl, list(SCL_INVALID_EXTRA)).mean() * 100),
                            }
                            if scl_stats_only['scl_cloud_shadow_pct'] > MAX_SCL_CLOUD_SHADOW_PCT:
                                rec.update(scl_stats_only)
                                rec.update({
                                    'accepted_s2_policy': False,
                                    'reject_reason': 'scl_cloud_shadow_pct_gt_30',
                                    'image_path': None,
                                    'scl_path': None,
                                    'scl_shape': list(scl.shape),
                                    'scl_dtype': 'uint8',
                                    'scl_unique_values': sorted(int(v) for v in np.unique(scl).tolist()),
                                })
                                records.append(rec)
                                continue

                            open_bands = {band: rasterio.open(find_band_file(scene_dir, band)) for band in BANDS}
                            try:
                                image = read_image_window(open_bands, ref_bounds)
                            finally:
                                for src in open_bands.values():
                                    src.close()
                            stats = tile_stats(image, scl)

                        shape_ok = image.shape == (TILE_SIZE, TILE_SIZE, len(BANDS)) and scl.shape == (TILE_SIZE, TILE_SIZE)
                        finite_ok = bool(np.isfinite(image).all())
                        accepted = bool(
                            shape_ok
                            and finite_ok
                            and stats['zero_pct'] < 99.0
                            and stats['image_std'] > 1e-6
                            and stats['scl_cloud_shadow_pct'] <= MAX_SCL_CLOUD_SHADOW_PCT
                        )
                        reasons = []
                        if not shape_ok:
                            reasons.append('shape_not_ok')
                        if not finite_ok:
                            reasons.append('non_finite')
                        if stats['zero_pct'] >= 99.0:
                            reasons.append('zero_pct_ge_99')
                        if stats['image_std'] <= 1e-6:
                            reasons.append('image_std_le_1e-6')
                        if stats['scl_cloud_shadow_pct'] > MAX_SCL_CLOUD_SHADOW_PCT:
                            reasons.append('scl_cloud_shadow_pct_gt_30')

                        if accepted and not img_path.exists():
                            np.save(img_path, image.astype('float32'))
                            np.save(out_scl_path, scl.astype('uint8'))

                        rec.update(stats)
                        rec.update({
                            'accepted_s2_policy': accepted,
                            'reject_reason': 'accepted' if accepted else '|'.join(reasons),
                            'image_path': str(img_path) if accepted else None,
                            'scl_path': str(out_scl_path) if accepted else None,
                            'image_shape': list(image.shape),
                            'scl_shape': list(scl.shape),
                            'image_dtype': 'float32',
                            'scl_dtype': 'uint8',
                            's2_bands': BANDS,
                            'scl_unique_values': sorted(int(v) for v in np.unique(scl).tolist()),
                        })
                    except Exception as exc:
                        rec.update({'accepted_s2_policy': False, 'reject_reason': f'{type(exc).__name__}: {exc}'})
                    records.append(rec)

                cells_for_date = remaining_cells

        # Cells not contained in any scene for this date.
        for cell in cells_for_date:
            records.append(make_base_rec(cell, date) | {'accepted_s2_policy': False, 'reject_reason': 'no_scene_window_inside'})

        log_progress('fin_fecha', {'date': date.date().isoformat(), **records_summary()})

log_progress('fin_generacion_pre_metadata', records_summary())
metadata = pd.DataFrame(records).drop_duplicates(subset=['tile_id'], keep='first').reset_index(drop=True)
metadata.shape


{"time": "2026-05-25T01:43:43.078972Z", "message": "inicio_generacion_tiles", "total_station_date_pairs": 1161, "n_dates": 129, "n_station_cells": 9}


{"time": "2026-05-25T01:43:43.086879Z", "message": "inicio_fecha", "date_idx": 1, "n_dates": 129, "date": "2020-01-02", "records": 0, "accepted": 0, "rejected": 0}


{"stage": "generate_station_tiles_fast", "i": 1, "n": 1161}


{"time": "2026-05-25T01:43:43.231086Z", "message": "fin_fecha", "date": "2020-01-02", "records": 9, "accepted": 3, "rejected": 6, "top_reject_reasons": {"scl_cloud_shadow_pct_gt_30": 6, "accepted": 3}}


{"time": "2026-05-25T01:43:43.236928Z", "message": "inicio_fecha", "date_idx": 2, "n_dates": 129, "date": "2020-01-07", "records": 9, "accepted": 3, "rejected": 6, "top_reject_reasons": {"scl_cloud_shadow_pct_gt_30": 6, "accepted": 3}}


...

{"time": "2026-05-25T01:50:52.459422Z", "message": "fin_fecha", "date": "2024-12-11", "records": 1152, "accepted": 611, "rejected": 541, "top_reject_reasons": {"accepted": 611, "scl_cloud_shadow_pct_gt_30": 415, "no_scene_window_inside": 108, "zero_pct_ge_99|image_std_le_1e-6": 18}}


{"time": "2026-05-25T01:50:52.471489Z", "message": "inicio_fecha", "date_idx": 129, "n_dates": 129, "date": "2024-12-16", "records": 1152, "accepted": 611, "rejected": 541, "top_reject_reasons": {"accepted": 611, "scl_cloud_shadow_pct_gt_30": 415, "no_scene_window_inside": 108, "zero_pct_ge_99|image_std_le_1e-6": 18}}


{"time": "2026-05-25T01:50:55.959323Z", "message": "fin_fecha", "date": "2024-12-16", "records": 1161, "accepted": 613, "rejected": 548, "top_reject_reasons": {"accepted": 613, "scl_cloud_shadow_pct_gt_30": 422, "no_scene_window_inside": 108, "zero_pct_ge_99|image_std_le_1e-6": 18}}


{"time": "2026-05-25T01:50:55.972100Z", "message": "fin_generacion_pre_metadata", "records": 1161, "accepted": 613, "rejected": 548, "top_reject_reasons": {"accepted": 613, "scl_cloud_shadow_pct_gt_30": 422, "no_scene_window_inside": 108, "zero_pct_ge_99|image_std_le_1e-6": 18}}


(1161, 38)

## Resumen y guardado


In [5]:
metadata_path = OUT / 'metadata_tiles_estaciones_scl.csv'
metadata_jsonl_path = OUT / 'metadata_tiles_estaciones_scl.jsonl'
summary_path = OUT / 'summary_tiles_estaciones_scl.json'
manifest_path = OUT / 'manifest_05_tiles_estaciones_filtrados_scl.json'

metadata.to_csv(metadata_path, index=False)
with metadata_jsonl_path.open('w', encoding='utf-8') as f:
    for rec in metadata.to_dict(orient='records'):
        f.write(json.dumps(rec, ensure_ascii=False, default=json_default) + '\n')

accepted = metadata[metadata['accepted_s2_policy'].fillna(False).astype(bool)].copy()
summary = {
    'notebook': '05_generar_tiles_estaciones_filtrados_scl.ipynb',
    'n_expected': int(len(station_cells) * len(dates)),
    'n_records': int(len(metadata)),
    'n_accepted': int(len(accepted)),
    'n_rejected': int(len(metadata) - len(accepted)),
    'accepted_fraction': float(len(accepted) / len(metadata)) if len(metadata) else 0.0,
    'n_station_cells': int(metadata['grid_id'].nunique()),
    'n_dates': int(metadata['date_day'].nunique()),
    'n_station_cells_with_8_accepted_dates': int(accepted.groupby('grid_id')['date_day'].nunique().ge(8).sum()) if len(accepted) else 0,
    'min_accepted_dates_per_cell': int(accepted.groupby('grid_id')['date_day'].nunique().min()) if len(accepted) else 0,
    'median_accepted_dates_per_cell': float(accepted.groupby('grid_id')['date_day'].nunique().median()) if len(accepted) else 0.0,
    'max_accepted_dates_per_cell': int(accepted.groupby('grid_id')['date_day'].nunique().max()) if len(accepted) else 0,
    'reject_reason_counts': {str(k): int(v) for k, v in metadata['reject_reason'].fillna('missing').value_counts().items()},
    'quality_policy': {
        'zero_pct_lt': 99.0,
        'image_std_gt': 1e-6,
        'scl_cloud_shadow_pct_lte': MAX_SCL_CLOUD_SHADOW_PCT,
        'scl_cloud_shadow_values': sorted(SCL_CLOUD_SHADOW),
        'scl_valid_visual_values': sorted(SCL_VALID_VISUAL),
        'scl_invalid_extra_values_recorded_only': sorted(SCL_INVALID_EXTRA),
    },
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=json_default), encoding='utf-8')
manifest = {
    'notebook': '05_generar_tiles_estaciones_filtrados_scl.ipynb',
    'description': 'Tiles Sentinel-2 64x64x12 para celdas de estación, filtrados con política SCL heredada de Situación 2.',
    'inputs': {k: str(v) for k, v in PATHS.items()},
    'input_md5': {k: md5_file(v) for k, v in PATHS.items() if v.exists() and v.is_file()},
    'outputs': {
        'images_dir': str(IMG_DIR),
        'scl_dir': str(SCL_DIR),
        'metadata_csv': str(metadata_path),
        'metadata_jsonl': str(metadata_jsonl_path),
        'summary': str(summary_path),
    },
    'summary': summary,
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=json_default), encoding='utf-8')
summary


{'notebook': '05_generar_tiles_estaciones_filtrados_scl.ipynb',
 'n_expected': 1161,
 'n_records': 1161,
 'n_accepted': 613,
 'n_rejected': 548,
 'accepted_fraction': 0.5279931093884582,
 'n_station_cells': 9,
 'n_dates': 129,
 'n_station_cells_with_8_accepted_dates': 9,
 'min_accepted_dates_per_cell': 52,
 'median_accepted_dates_per_cell': 67.0,
 'max_accepted_dates_per_cell': 79,
 'reject_reason_counts': {'accepted': 613,
  'scl_cloud_shadow_pct_gt_30': 422,
  'no_scene_window_inside': 108,
  'zero_pct_ge_99|image_std_le_1e-6': 18},
 'quality_policy': {'zero_pct_lt': 99.0,
  'image_std_gt': 1e-06,
  'scl_cloud_shadow_pct_lte': 30.0,
  'scl_cloud_shadow_values': [3, 8, 9, 10],
  'scl_valid_visual_values': [4, 5, 6, 7],
  'scl_invalid_extra_values_recorded_only': [0, 1, 2, 11]}}